# 06 - Hybrid Ensemble + Evaluation

Combines the classical Random Forest (04) and the quantum kernel SVM (05) into a simple hybrid vote, compares all three side by side on the same held-out sample using the real objectives/metrics registry, then runs a robustness spot-check (speckle/noise perturbations) against the classical model on a full chip.

In [1]:
import sys
sys.path.insert(0, r"d:\project-raw-data\sphoorthq-geoverse")

import pickle
import numpy as np

from src.ai.classic.sen1floods11_dataset import load_split, chip_id_from_s1_filename, read_s1, read_label
from src.fusion.pixel_features import build_feature_cube, cube_to_pixel_table
from src.qml.hybrid_classifier import QuantumKernelSVM, QuantumBackend
from src.qml import ibm_quantum
from src.ai.objectives.registry import evaluate
from src.ai.robustness.perturbations import add_speckle_noise, add_gaussian_noise
from src.observability.run_logger import RunLogger

logger = RunLogger("06_hybrid_ensemble_evaluation")
RNG = np.random.default_rng(11)

## Rebuild the same held-out sample as notebook 05

(Same chip, same seed, same N_TRAIN=20/N_TEST=10 - now practical because `compute_gram_matrix` batches every circuit for a gram matrix into one Qiskit Runtime job instead of one job per pair, see notebook 05. So the comparison is apples-to-apples with notebook 05's quantum-only results.)

In [2]:
N_TRAIN = 20
N_TEST = 10

with logger.stage("rebuild_sample") as stage:
    pairs = load_split("train")
    s1_filename, _ = pairs[3]
    chip_id = chip_id_from_s1_filename(s1_filename)
    s1 = read_s1(chip_id)
    label = read_label(chip_id)
    cube = build_feature_cube(s1)
    x_all, y_all = cube_to_pixel_table(cube, label, raw_s1=s1)

    sample_rng = np.random.default_rng(7)
    water_idx = np.where(y_all == 1)[0]
    land_idx = np.where(y_all == 0)[0]
    n_each = (N_TRAIN + N_TEST) // 2
    chosen = np.concatenate([
        sample_rng.choice(water_idx, size=n_each, replace=False),
        sample_rng.choice(land_idx, size=n_each, replace=False),
    ])
    sample_rng.shuffle(chosen)
    x_sample, y_sample = x_all[chosen], y_all[chosen]
    x_train, y_train = x_sample[:N_TRAIN], y_sample[:N_TRAIN]
    x_test, y_test = x_sample[N_TRAIN:N_TRAIN + N_TEST], y_sample[N_TRAIN:N_TRAIN + N_TEST]
    stage.metrics = {"n_train": len(y_train), "n_test": len(y_test)}

[06_hybrid_ensemble_evaluation] -> rebuild_sample ...
[06_hybrid_ensemble_evaluation] <- rebuild_sample [OK] 0.053s {'n_train': 20, 'n_test': 10}


In [3]:
FORCE_SIMULATION = True  # set False to use the live IBM Quantum account (see config/platform.yaml)

with logger.stage("load_classical_model") as stage:
    with open(r"d:\project-raw-data\sphoorthq-geoverse\datasets\processed\models\classical_rf_v1.pkl", "rb") as f:
        rf = pickle.load(f)
    y_pred_classical = rf.predict(x_test)
    metrics_classical = evaluate("flood-segmentation", y_pred_classical, y_test)

with logger.stage("train_quantum_kernel") as stage:
    ibm_service = None if FORCE_SIMULATION else ibm_quantum.get_ibm_service()
    q_model = QuantumKernelSVM(backend=QuantumBackend.IBM, service=ibm_service)
    q_model.fit(x_train, y_train)
    y_pred_quantum = q_model.predict(x_test)
    metrics_quantum = evaluate("flood-segmentation", y_pred_quantum, y_test)
    stage.metrics = {"forced_simulation": FORCE_SIMULATION, "is_real_hardware": ibm_service is not None}

with logger.stage("hybrid_vote") as stage:
    # simple hybrid: water if either model says water (favors recall - fewer missed flood pixels)
    y_pred_hybrid = ((y_pred_classical == 1) | (y_pred_quantum == 1)).astype(int)
    metrics_hybrid = evaluate("flood-segmentation", y_pred_hybrid, y_test)
    stage.metrics = {k: round(v, 4) for k, v in metrics_hybrid.items()}

[06_hybrid_ensemble_evaluation] -> load_classical_model ...


[06_hybrid_ensemble_evaluation] <- load_classical_model [OK] 0.173s {}
[06_hybrid_ensemble_evaluation] -> train_quantum_kernel ...


[06_hybrid_ensemble_evaluation] <- train_quantum_kernel [OK] 56.163s {'forced_simulation': True, 'is_real_hardware': False}
[06_hybrid_ensemble_evaluation] -> hybrid_vote ...
[06_hybrid_ensemble_evaluation] <- hybrid_vote [OK] 0.0s {'iou': 0.6667, 'f1': 0.8, 'precision': 1.0, 'recall': 0.6667, 'boundary_f1': 1.0}


In [4]:
import pandas as pd

comparison = pd.DataFrame({
    "classical_rf": metrics_classical,
    "quantum_kernel": metrics_quantum,
    "hybrid_vote": metrics_hybrid,
}).T
print(comparison.round(4))

                   iou   f1  precision  recall  boundary_f1
classical_rf    0.0000  0.0        1.0  0.0000          0.0
quantum_kernel  0.6667  0.8        1.0  0.6667          1.0
hybrid_vote     0.6667  0.8        1.0  0.6667          1.0


## Robustness spot-check

Runs the classical model against a full chip under clean vs. perturbed conditions (heavier speckle, Gaussian noise) - same perturbation harness as `src/ai/robustness/`, applied here directly rather than through the full `run_robustness_suite` (which expects a segmentation-mask predictor, not a per-pixel classifier).

In [5]:
def predict_chip_with_rf(s1_db: np.ndarray, model) -> np.ndarray:
    cube = build_feature_cube(s1_db)
    channels, h, w = cube.shape
    flat = cube.reshape(channels, h * w).T
    return model.predict(flat).reshape(h, w)

with logger.stage("robustness_speckle") as stage:
    clean_pred = predict_chip_with_rf(s1, rf)
    clean_metrics = evaluate("flood-segmentation", clean_pred[label != -1], label[label != -1])

    noisy_s1 = s1.copy()
    noisy_s1[0] = add_speckle_noise(s1[0], looks=1, rng=RNG)
    noisy_s1[1] = add_speckle_noise(s1[1], looks=1, rng=RNG)
    noisy_pred = predict_chip_with_rf(noisy_s1, rf)
    noisy_metrics = evaluate("flood-segmentation", noisy_pred[label != -1], label[label != -1])

    iou_drop = clean_metrics["iou"] - noisy_metrics["iou"]
    stage.metrics = {"clean_iou": round(clean_metrics["iou"], 4), "noisy_iou": round(noisy_metrics["iou"], 4), "iou_drop": round(iou_drop, 4)}

print(f"Clean IoU: {clean_metrics['iou']:.4f}  |  Heavy-speckle IoU: {noisy_metrics['iou']:.4f}  |  Drop: {iou_drop:.4f}")

[06_hybrid_ensemble_evaluation] -> robustness_speckle ...


[06_hybrid_ensemble_evaluation] <- robustness_speckle [OK] 2.442s {'clean_iou': 0.0872, 'noisy_iou': 0.0194, 'iou_drop': 0.0678}
Clean IoU: 0.0872  |  Heavy-speckle IoU: 0.0194  |  Drop: 0.0678


In [6]:
logger.log_metrics({
    "classical_iou": metrics_classical["iou"],
    "quantum_iou": metrics_quantum["iou"],
    "hybrid_iou": metrics_hybrid["iou"],
    "robustness_iou_drop": iou_drop,
})
logger.finalize()

[06_hybrid_ensemble_evaluation] run complete in 58.887s -> D:\project-raw-data\sphoorthq-geoverse\datasets\reports\runs\24074435-9717-4286-8d3f-65710b9895f5.json


'D:\\project-raw-data\\sphoorthq-geoverse\\datasets\\reports\\runs\\24074435-9717-4286-8d3f-65710b9895f5.json'